# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya — Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the metadata and the Croissant structure.

Below we print all available record sets (referenced by their `@id`) and their core fields.

In [ ]:
# Get available record sets by @id
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in the Croissant package.")
else:
    print("Available record sets:")
    for rec_set in record_sets:
        print(f"- @id: {rec_set.id} | name: {getattr(rec_set, 'name', '(none)')}")
        print("  Fields:")
        for fld in rec_set.fields:
            print(f"    - @id: {fld.id} | name: {getattr(fld, 'name', '(none)')} | type: {getattr(fld, 'data_type', '(none)')}")
    print()
    # Show example records of the first record set
    sample_rec_set = record_sets[0]
    print(f"Sample records for record set {sample_rec_set.id}:")
    for i, rec in enumerate(dataset.records(record_set=sample_rec_set.id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis.
All record sets and their fields must be referenced by their unique `@id`.

We demonstrate how to extract all records from each record set into a dictionary of DataFrames below.

In [ ]:
# Extract all available record sets into pandas DataFrames
dataframes = {}
record_set_ids = [rec_set.id for rec_set in record_sets]

for record_set_id in record_set_ids:
    recs = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(recs)
    print(f"Loaded DataFrame for record set {record_set_id} with shape {dataframes[record_set_id].shape}")

if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nColumns in DataFrame for record set {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    print(f"\nPreview of data:")
    display(dataframes[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalization, and grouping. All references use their exact `@id` values.

Below, we demonstrate EDA on a numeric field from the first available record set, if present.

In [ ]:
# Example: perform filtering and normalization on a numeric field of the first record set
import numpy as np

if record_set_ids:
    df = dataframes[first_rs_id]
    print(f"Available columns in record set {first_rs_id}:")
    print(df.columns.tolist())
    
    # Try to pick the first numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_cols:
        numeric_field_id = numeric_cols[0]  # Using the field's @id as column name
        print(f"\nChosen numeric field for EDA: {numeric_field_id}")
        # Filter
        threshold = df[numeric_field_id].quantile(0.95)  # Use 95th percentile as an example
        filtered_df = df[df[numeric_field_id] < threshold].copy()
        print(f"Filtered outliers above {threshold:.2f} in {numeric_field_id}.")
        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
            filtered_df[numeric_field_id].std()
        )
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Group if possible
        # Choose a non-numeric field as group
        non_num_cols = [c for c in df.columns if c not in numeric_cols and df[c].dtype == 'object']
        if non_num_cols:
            group_field_id = non_num_cols[0]
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric columns found to perform EDA.")
else:
    print("No record sets to perform EDA.")

## 5. Visualization
Visualize the distribution of a numeric field and relationships, referencing fields by their `@id`.

Below is an example histogram and boxplot for a numeric field (if present).

In [ ]:
import matplotlib.pyplot as plt

if record_set_ids and numeric_cols:
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.hist(df[numeric_field_id].dropna(), bins=40, color='skyblue', edgecolor='k')
    plt.title(f'Histogram of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    
    plt.subplot(1,2,2)
    plt.boxplot(df[numeric_field_id].dropna())
    plt.title(f'Boxplot of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion

- We loaded the FAIR<sup>2</sup> Croissant dataset and explored its metadata and record sets using unique `@id`s.
- Data was extracted and inspected using pandas DataFrames for each record set.
- Exploratory Data Analysis demonstrated numeric filtering, normalization, basic statistics, and visualization.
- The dataset, based on survey and regression outputs, provides insight into knowledge adoption in rangeland management in Northern Kenya.
- For advanced analysis, further domain knowledge and detailed use of metadata would enhance interpretation and enable broader comparisons.